In [1]:
import uproot
import numpy as np
import torch
import torch.nn as nn
import matplotlib.pyplot as plt

# ---------------- MODEL ----------------
class PeakNet(nn.Module):
    def __init__(self):
        super().__init__()
        self.net = nn.Sequential(
            nn.Conv1d(1, 16, 5, padding=2),
            nn.ReLU(),
            nn.Conv1d(16, 32, 5, padding=2),
            nn.ReLU(),
            nn.Conv1d(32, 1, 3, padding=1),
            nn.Sigmoid()
        )

    def forward(self, x):
        return self.net(x)

# ---------------- LOAD ROOT HIST ----------------
def load_root_hist(file, hist_name):
    f = uproot.open(file)
    hist = f[hist_name]

    values = hist.values()
    return values

# ---------------- LABEL GENERATION ----------------
def make_labels(n_bins, peak_positions, width=5):
    labels = np.zeros(n_bins)

    for p in peak_positions:
        for i in range(-width, width+1):
            idx = int(p + i)
            if 0 <= idx < n_bins:
                labels[idx] = 1

    return labels

# ---------------- NORMALIZATION ----------------
def normalize(y):
    return (y - np.mean(y)) / (np.std(y) + 1e-8)

# ---------------- TRAIN ----------------
def train_model(model, spectra, labels_list, epochs=10):
    optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)
    loss_fn = nn.BCELoss()

    for epoch in range(epochs):
        total_loss = 0

        for y, labels in zip(spectra, labels_list):

            y = normalize(y)

            x_tensor = torch.tensor(y, dtype=torch.float32).unsqueeze(0).unsqueeze(0)
            y_tensor = torch.tensor(labels, dtype=torch.float32).unsqueeze(0).unsqueeze(0)

            pred = model(x_tensor)
            loss = loss_fn(pred, y_tensor)

            optimizer.zero_grad()
            loss.backward()
            optimizer.step()

            total_loss += loss.item()

        print(f"Epoch {epoch}: Loss = {total_loss:.4f}")

# ---------------- PEAK FINDING ----------------
def find_peaks_ml(model, hist_array, threshold=0.5):
    hist_array = normalize(hist_array)

    x = torch.tensor(hist_array, dtype=torch.float32).unsqueeze(0).unsqueeze(0)

    with torch.no_grad():
        output = model(x).squeeze().numpy()

    indices = np.where(output > threshold)[0]

    peaks = []
    if len(indices) > 0:
        group = [indices[0]]
        for i in indices[1:]:
            if i - group[-1] <= 2:
                group.append(i)
            else:
                peaks.append(int(np.mean(group)))
                group = [i]
        peaks.append(int(np.mean(group)))

    return peaks, output

# ---------------- MAIN ----------------
if __name__ == "__main__":

    # ---- Load your real spectra ----
    spectra = []
    labels_list = []

    # Example: multiple ROOT files
    files = ["gamma_spec_main_1000ms.root","gamma_spec_main_1s_2s.root","gamma_spec_main_2s_3s.root","gamma_spec_main_3s_4s.root","gamma_spec_main_4s_5s.root"]

    for f in files:
        y = load_root_hist(f, "hist")

        # Replace with TSpectrum output or manual list
        known_peaks = [511, 662]  

        labels = make_labels(len(y), known_peaks)

        spectra.append(y)
        labels_list.append(labels)

    # ---- Train ----
    model = PeakNet()
    train_model(model, spectra, labels_list, epochs=15)

    torch.save(model.state_dict(), "peak_model.pth")

    # ---- Test on one spectrum ----
    test_hist = spectra[0]

    peaks, output = find_peaks_ml(model, test_hist)

    print("Predicted peaks:", peaks)

    # ---- Plot ----
    plt.plot(test_hist, label="Spectrum")

    for p in peaks:
        plt.axvline(p, linestyle="--")

    plt.legend()
    plt.show()

    # ---- Save peaks for ROOT ----
    np.savetxt("peaks.txt", peaks, fmt="%d")

FileNotFoundError: [Errno 2] No such file or directory: 'peak_model.pth'

In [ ]:
peaks = find_peaks_ml(hist_array)

np.savetxt("peaks.txt", peaks, fmt="%d")